# Nuclei Clustering Pipeline - Adaptive Tree Exploration

**Pipeline explorador con árbol de clustering adaptativo**

Este notebook orquesta un pipeline explorador que:
1. Construye un árbol de clustering binario
2. En cada paso, divide CADA rama en k=2 (sin decisiones frágiles)
3. Calcula métricas para cada rama (excepto paso 0)
4. Reporta la mejor rama según F1 score

Ventajas para investigación:
- Explora todas las combinaciones posibles
- No depende de funciones de decisión frágiles
- Muestra el árbol completo de descubrimiento
- Ideal para análisis exploratorio

In [1]:
# CELDA 1: Setup - Importar librerías y configurar entorno

import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("⏳ Importando módulos...")

# Limpiar cache de módulos para asegurar recarga
modules_to_remove = [key for key in sys.modules.keys() if key.startswith('utils') or key == 'process_image']
for mod in modules_to_remove:
    del sys.modules[mod]

# Importar el pipeline
from process_image import ImageProcessingPipeline

print("✅ Setup completado")

⏳ Importando módulos...
✅ Setup completado


In [2]:
# CELDA 2: Configurar rutas y parámetros

# ====================== CONFIGURACIÓN DE RUTAS ======================
# OPCIÓN 1: Local (Windows)
JSON_PATH = r'C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train\_annotations.coco.json'
IMAGES_DIR = r'C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train'

# OPCIÓN 2: Kaggle (descomentar si estás en Kaggle)
# JSON_PATH = '/kaggle/input/tu-dataset/train/_annotations.coco.json'
# IMAGES_DIR = '/kaggle/input/tu-dataset/train'

# ====================== PARÁMETROS DEL PIPELINE ======================
IMAGE_INDEX = 0  # Cambia este número para procesar otras imágenes (0 a N-1)
N_STEPS = 6  # Número de pasos para crecer el árbol (ej: 2, 3, 4, 5)
MODEL_NAME = 'uni2'  # Opciones: 'biomedclip', 'uni', 'optimus', 'uni2'
PREPROCESSING_METHOD = 'none'  # Opciones: 'clahe', 'equalize', 'normalize', 'none'
CATEGORY_IDS = [4, 5]  # IDs de categorías a incluir (kernels)
VISUALIZE = True  # Mostrar gráficos durante el procesamiento

# ====================== ESTRATEGIA DE EVALUACIÓN ======================
EVALUATION_STRATEGY = 'segmentation'  # Opciones: 'segmentation', 'bbox'
# - 'segmentation': Usa máscaras de segmentación, métricas DICE, IoU, pixel-wise
# - 'bbox': Método legacy con grouping de patches y overlap de bounding boxes

print(f"📂 Configuración:")
print(f"   JSON: {JSON_PATH}")
print(f"   Imágenes: {IMAGES_DIR}")
print(f"   Imagen a procesar: {IMAGE_INDEX}")
print(f"   Pasos del árbol: {N_STEPS}")
print(f"   Modelo: {MODEL_NAME}")
print(f"   Preprocesamiento: {PREPROCESSING_METHOD}")
print(f"   Estrategia de evaluación: {EVALUATION_STRATEGY}")
print(f"   Visualizar: {VISUALIZE}")

# Verificar que existen las rutas
assert os.path.exists(JSON_PATH), f"❌ JSON no encontrado: {JSON_PATH}"
assert os.path.isdir(IMAGES_DIR), f"❌ Directorio no encontrado: {IMAGES_DIR}"
print("\n✅ Rutas verificadas")

📂 Configuración:
   JSON: C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train\_annotations.coco.json
   Imágenes: C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train
   Imagen a procesar: 0
   Pasos del árbol: 6
   Modelo: uni2
   Preprocesamiento: none
   Estrategia de evaluación: segmentation
   Visualizar: True

✅ Rutas verificadas


In [3]:
# CELDA 3: Ejecutar Pipeline del Árbol de Clustering

# Crear instancia del pipeline con los parámetros configurados
print("⏳ Inicializando pipeline...")
pipeline = ImageProcessingPipeline(
    json_path=JSON_PATH,
    images_dir=IMAGES_DIR,
    model_name=MODEL_NAME,
    preprocessing_method=PREPROCESSING_METHOD,
    category_ids=CATEGORY_IDS,
    visualize=VISUALIZE,
    n_steps=N_STEPS,  # Número de pasos para crecer el árbol
    evaluation_strategy=EVALUATION_STRATEGY  # Estrategia de evaluación
)

# Ejecutar todo el pipeline (construye árbol con N pasos)
tree = pipeline.run(IMAGE_INDEX)

# Verificar que se completó exitosamente
if tree:
    print("\n🎉 Pipeline ejecutado exitosamente!")

⏳ Inicializando pipeline...
⏳ Cargando modelo...
Cargando UNI 2 (mahmoodlab/uni-v2)...
⚠️ Error cargando UNI 2: 401 Client Error. (Request ID: Root=1-69933f2c-4aba2e6239d99d372a58f7b6;51ba91d1-67e4-491d-a2f4-ea37e5220c50)

Repository Not Found for url: https://huggingface.co/mahmoodlab/uni-v2/resolve/main/pytorch_model.bin.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69933f2c-4aba2e6239d99d372a58f7b6;51ba91d1-67e4-491d-a2f4-ea37e5220c50)

Repository Not Found for url: https://huggingface.co/mahmoodlab/uni-v2/resolve/main/pytorch_model.bin.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
# CELDA 4: Inspeccionar el Árbol de Clustering

print("🔍 INFORMACIÓN DEL ÁRBOL DE CLUSTERING:\n")

# Información básica
print(f"Imagen procesada: {pipeline.fname}")
print(f"Dimensiones: {pipeline.W}x{pipeline.H}")

# Detectar tipo de ground truth
if hasattr(pipeline, 'ground_truth'):
    if isinstance(pipeline.ground_truth, dict):
        if 'mask' in pipeline.ground_truth or 'masks' in pipeline.ground_truth:
            print(f"Ground truth: Máscaras de segmentación ({pipeline.ground_truth['n_instances']} instancias)")
        else:
            print(f"Número de anotaciones GT: {len(pipeline.ground_truth)}")
    else:
        print(f"Número de anotaciones GT: {len(pipeline.ground_truth)}")
else:
    print(f"Número de anotaciones GT: {len(pipeline.boxes)}")

print()

# Información del árbol
leaves = tree.root.get_all_leaves()
print("📊 ÁRBOL:")
print(f"  Total de nodos: {len(tree.all_nodes)}")
print(f"  Ramas finales (hojas): {len(leaves)}")
print(f"  Pasos ejecutados: 0-{leaves[0].step if leaves else '?'}\n")

# Obtener mejor nodo (puede ser de cualquier paso)
best_node = tree.get_best_node()
best_leaf = tree.get_best_leaf()

# Mostrar métricas de cada rama (solo hojas)
print("📈 MÉTRICAS POR RAMA - HOJAS FINALES (ordenadas por F1 Score):\n")

leaves_with_metrics = [(l.node_id, l.step, l.n_patches, l.metrics) for l in leaves if l.metrics is not None]
leaves_with_metrics.sort(key=lambda x: x[3]['f1_coverage'], reverse=True)

if leaves_with_metrics:
    for i, (node_id, step, n_patches, metrics) in enumerate(leaves_with_metrics, 1):
        print(f"  {i}. Rama {node_id} (Paso {step}, {n_patches} patches):")
        
        # Detectar tipo de estrategia basándose en las métricas disponibles
        if 'groups_TP' in metrics:
            # Estrategia bbox
            print(f"     TP: {metrics['groups_TP']:2d} | FP: {metrics['groups_FP']:2d}")
            print(f"     Precisión: {metrics['group_precision']:.3f} | Recall: {metrics['gt_recall_coverage']:.3f}")
        elif 'dice' in metrics:
            # Estrategia segmentation
            print(f"     DICE: {metrics['dice']:.3f} | IoU: {metrics['iou']:.3f}")
            print(f"     Precisión: {metrics['precision']:.3f} | Recall: {metrics['recall']:.3f}")
        
        print(f"     F1 Score: {metrics['f1_coverage']:.3f}\n")
    
    # Mejor rama (hoja)
    print(f"🏆 MEJOR RAMA (HOJA): {best_leaf.node_id}")
    print(f"   F1 Score: {best_leaf.metrics['f1_coverage']:.3f}")
    print(f"   Paso: {best_leaf.step}, Patches: {best_leaf.n_patches}")
    
    # Mostrar métricas específicas según estrategia
    if 'groups_TP' in best_leaf.metrics:
        print(f"   TP: {best_leaf.metrics['groups_TP']}, FP: {best_leaf.metrics['groups_FP']}")
    elif 'dice' in best_leaf.metrics:
        print(f"   DICE: {best_leaf.metrics['dice']:.3f}, IoU: {best_leaf.metrics['iou']:.3f}")
else:
    print("  ⚠️ No hay ramas con métricas (el árbol solo tiene paso 0)")

# Advertencia si el mejor nodo overall está en un paso intermedio
if best_node and best_node != best_leaf:
    print("\n⚠️ ⭐ IMPORTANTE:")
    print(f"   El MEJOR NODO OVERALL está en paso {best_node.step}, no en las hojas!")
    print(f"   Nodo: {best_node.node_id}, F1: {best_node.metrics['f1_coverage']:.3f}")
    print(f"   Recomendación: Ver Celda 4B para análisis detallado y recomendación de pasos.")


In [ ]:
# CELDA 4B: Análisis de Métricas por Paso (Tabla) + Recomendación

print("📊 ESTADÍSTICAS POR PASO DEL ÁRBOL:\n")

# Obtener estadísticas por paso
step_stats = tree.get_metrics_by_step()

# Mostrar en tabla
print(f"{'Paso':<6} {'Nodos':<8} {'Evaluados':<12} {'F1 Min':<10} {'F1 Media':<10} {'F1 Max':<10}")
print("-"*80)

for step in sorted(step_stats.keys()):
    stats = step_stats[step]
    f1_min_str = f"{stats['f1_min']:.3f}" if stats['f1_min'] is not None else "—"
    f1_mean_str = f"{stats['f1_mean']:.3f}" if stats['f1_mean'] is not None else "—"
    f1_max_str = f"{stats['f1_max']:.3f}" if stats['f1_max'] is not None else "—"
    
    print(f"{step:<6} {stats['n_nodes']:<8} {stats['n_evaluated']:<12} "
          f"{f1_min_str:<10} {f1_mean_str:<10} {f1_max_str:<10}")

print("="*80)

# Análisis e insights + RECOMENDACIÓN
print("\n💡 INSIGHTS:")
print("  - Paso 0: Sin evaluación (separación inicial fondo/tejido)")

for step in sorted(step_stats.keys()):
    if step == 0:
        continue
    
    stats = step_stats[step]
    if stats['f1_mean'] is not None:
        if stats['f1_max'] >= 0.7:
            quality = "✅ Excelente"
        elif stats['f1_max'] >= 0.5:
            quality = "⚠️ Aceptable"
        else:
            quality = "❌ Pobre"
        
        print(f"  - Paso {step}: {stats['n_nodes']} nodos ({stats['n_evaluated']} evaluados) - "
              f"F1 promedio: {stats['f1_mean']:.3f} {quality}")

# RECOMENDACIÓN DE PASOS ÓPTIMOS
optimal_steps = tree.get_optimal_steps()
best_node = tree.get_best_node()

print("\n" + "="*80)
print("⭐ RECOMENDACIÓN")
print("="*80)

if best_node and optimal_steps is not None:
    leaves = tree.root.get_all_leaves()
    best_leaf_f1 = max([l.metrics['f1_coverage'] for l in leaves if l.metrics is not None], default=0)
    
    print(f"\n🏆 MEJOR NODO ENCONTRADO: {best_node.node_id}")
    print(f"   - Paso: {optimal_steps}")
    print(f"   - F1 Score: {best_node.metrics['f1_coverage']:.3f}")
    print(f"   - Patches: {best_node.n_patches}")
    
    if optimal_steps < N_STEPS:
        print(f"\n✅ RECOMENDACIÓN: N_STEPS={optimal_steps} es SUFICIENTE")
        print(f"   No necesitas ejecutar hasta {N_STEPS} pasos.")
        print(f"   El mejor resultado está en paso {optimal_steps} con F1={best_node.metrics['f1_coverage']:.3f}")
        print(f"   (vs mejor hoja en paso {N_STEPS} con F1={best_leaf_f1:.3f})")
    else:
        print(f"\n ℹ️ El mejor nodo está en el último paso ({optimal_steps}).")
        print(f"   Podría intentar N_STEPS={optimal_steps + 1} para explorar más.")
else:
    print("⚠️ No hay suficientes datos para hacer recomendación")


In [ ]:
# CELDA 4C: Visualizar Árbol Completo (Gráficos)

print("🌳 VISUALIZANDO ESTRUCTURA DEL ÁRBOL CON MÉTRICAS...\n")
tree.visualize_tree_structure()

In [ ]:
# CELDA 5: Visualizar el Mejor Nodo (de cualquier paso) con Grouping y Comparativa

# Obtener el mejor nodo (puede ser de cualquier paso, no solo el último)
best_node = tree.get_best_node()

if best_node and best_node.metrics:
    print("🔬 ANÁLISIS DEL MEJOR NODO:\n")
    print(f"Node ID: {best_node.node_id}")
    print(f"Paso: {best_node.step}")
    print(f"Patches: {best_node.n_patches}")
    print(f"F1 Score: {best_node.metrics['f1_coverage']:.3f}\n")
    
    print("Métricas completas:")
    for key, value in best_node.metrics.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.3f}")
        else:
            print(f"  {key}: {value}")
    
    # Visualizar con grouping y comparativa con GT
    print("\n📊 Visualizando mejor nodo con detección de núcleos y comparativa con GT...\n")
    tree.visualize_best_node_with_grouping()
    
else:
    print("⚠️ No hay nodo con métricas para analizar")

# Para todas las imagenes

In [ ]:
# CELDA 6: Generar Reporte HTML para Todas las Imágenes

print("📄 GENERANDO REPORTE PARA TODAS LAS IMÁGENES\n")
print("="*80)

# Recargar módulo para asegurar que tiene la función generate_html_report
import importlib
import process_image
importlib.reload(process_image)

from process_image import generate_html_report

# Generar reporte
output_report = "reporte_clustering.html"

try:
    generate_html_report(
        json_path=JSON_PATH,
        images_dir=IMAGES_DIR,
        output_path=output_report,
        n_steps=N_STEPS,
        model_name=MODEL_NAME,
        preprocessing_method=PREPROCESSING_METHOD,
        category_ids=CATEGORY_IDS,
        evaluation_strategy=EVALUATION_STRATEGY  # Estrategia de evaluación
    )
    
    # Verificar que el archivo existe
    import os
    if os.path.exists(output_report):
        file_size = os.path.getsize(output_report) / (1024 * 1024)  # MB
        print(f"\n🎉 ¡REPORTE GENERADO EXITOSAMENTE!")
        print(f"📁 Archivo: {output_report} ({file_size:.2f} MB)")
        print(f"📂 Ubicación: {os.path.abspath(output_report)}")
        print(f"\nAbre el archivo en tu navegador para ver los resultados.")
    else:
        print(f"\n⚠️ El archivo {output_report} no se creó correctamente")

except Exception as e:
    print(f"\n❌ Error durante la generación del reporte:")
    print(f"{type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


## Instrucciones de Uso - Pipeline Explorador

### Pasos para ejecutar (análisis de una imagen):

1. **Celda 1**: Ejecuta el setup inicial para importar módulos
2. **Celda 2**: Ajusta los parámetros según tus necesidades
3. **Celda 3**: Ejecuta el pipeline que construye el árbol (N pasos)
4. **Celda 4**: Inspecciona las métricas de todas las ramas
5. **Celda 4B**: (Opcional) Tabla de estadísticas agregadas por paso + Recomendación
6. **Celda 4C**: (Opcional) Gráficos de evolución de métricas y árbol completo
7. **Celda 5**: (Opcional) Analiza en detalle la mejor rama

### Para generar un reporte HTML con TODAS las imágenes:

**Ejecuta Celda 6**: Genera un reporte HTML interactivo que contiene:
- Resumen global con estadísticas agregadas
- Tabla resumen para cada imagen
- Visualización detallada de cada imagen con detecciones vs GT
- Recomendaciones automáticas de pasos óptimos
- Conclusiones y análisis comparativo

### Parámetros configurables (Celda 2):

- `IMAGE_INDEX`: Qué imagen procesar (0 para la primera)
- `N_STEPS`: Número de pasos para crecer el árbol (ej: 2, 3, 4, 5)
  - Paso 0: División inicial (Fondo vs Tejido) - sin evaluación
  - Pasos 1+: Se dividen TODAS las ramas en k=2 - CON evaluación
- `MODEL_NAME`: Modelo de embeddings (`biomedclip`, `uni`, `optimus`, `uni2`)
- `PREPROCESSING_METHOD`: (`clahe`, `equalize`, `normalize`, `none`)
- `VISUALIZE`: Mostrar gráficos

### Cómo funciona el árbol:

La imagen muestra un árbol binario que crece en pasos:
- **Paso 0**: 1 rama → 2 clusters
- **Paso 1**: 2 ramas → cada una se divide en 2 (4 hojas total)
- **Paso 2**: 4 ramas → cada una se divide en 2 (8 hojas total)
- **Paso 3**: 8 ramas → cada una se divide en 2 (16 hojas total)
- **Paso N**: 2^N hojas diferentes

### Interpretación de resultados:

- **Ramas con F1 alto**: Buena separación de núcleos
- **Ramas con F1 bajo**: Mala separación (posiblemente todas nucleares o todo fondo)
- **Mejor rama**: La que maximiza F1 Score (Precisión vs Recall)

### Para procesar otra imagen:

Solo cambia `IMAGE_INDEX` en la Celda 2 y re-ejecuta las celdas 3, 4 y 5.

### Para experimentar más pasos:

Cambia `N_STEPS` a 2, 3, 4, 5, etc. Tendrás más ramas pero también más tiempo de cómputo.

### Notas importantes:

- Paso 0 NO tiene evaluación (es separación fondo/tejido)
- Solo pasos 1+ se evalúan con métricas
- Las métricas se calculan con criterio "OVERLAP" (cualquier solapamiento cuenta)
- Utiliza Python con GPU si disponible para más velocidad

## 🎯 Cambio importante: Mejor Nodo en Cualquier Paso

**Ahora el pipeline puede identificar el mejor resultado en CUALQUIER paso**, no solo en el último.

### Ejemplo:
Si ejecutas N_STEPS=4, el árbol crece así:
- Paso 0: 1 nodo
- Paso 1: 2 nodos (posible mejor: F1=0.85)  ← **PUEDE ser el mejor!**
- Paso 2: 4 nodos  (posible mejor: F1=0.88)  ← **PUEDE ser el mejor!**
- Paso 3: 8 nodos  (posible mejor: F1=0.80)
- Paso 4: 16 nodos (posible mejor: F1=0.75)

**Resultado**: Paso 2 sería el óptimo (F1=0.88), aunque ejecutaste 4 pasos.

### Cómo funciona:
1. **Celda 4**: Busca el mejor nodo en **hojas finales** (nodos que no fueron divididos más)
2. **Celda 4B**: Busca el mejor nodo **en TODOS los pasos** y recomienda el N_STEPS óptimo
3. **Celda 5**: Visualiza el mejor nodo (sea del paso que sea)

### Recomendación automática:
Si el mejor nodo está en paso 2, verás:
```
✅ RECOMENDACIÓN: N_STEPS=2 es SUFICIENTE
No necesitas ejecutar hasta 4 pasos.
```

## 📄 Generación de Reportes

### Con la Celda 6 - Reporte HTML Interactivo

Puedes generar un reporte HTML completo que procesa **TODAS las imágenes del dataset** en una sola ejecución.

**Ventajas:**
- ✅ Procesa todas las imágenes automáticamente
- ✅ Genera visualizaciones de todas ellas
- ✅ Estadísticas globales y comparativas
- ✅ Recomendaciones de pasos óptimos por imagen
- ✅ Salida en archivo HTML interactivo (no depende de la notebook)
- ✅ Se puede compartir y abrir en navegador

**Resultado:**
Un archivo `reporte_clustering.html` con:
1. **Resumen General**: Estadísticas agregadas (F1 promedio, TP, FP, etc.)
2. **Tabla Comparativa**: Métrica para cada imagen
3. **Detalle Visual**: Visualización de cada imagen con su detección
4. **Recomendaciones**: Sugerencias basadas en resultados

### Ejemplo de salida:
```
📂 Total de imágenes en dataset: 10

⏳ Procesando 10 imágenes...
[1/10] Procesando imagen... ✅ F1: 0.875 | Pasos óptimos: 2
[2/10] Procesando imagen... ✅ F1: 0.812 | Pasos óptimos: 3
...
[10/10] Procesando imagen... ✅ F1: 0.768 | Pasos óptimos: 2

✅ Reporte guardado en: reporte_clustering.html
```

### Parámetros desde terminal (alternativa):

Si prefieres ejecutar desde terminal directamente sin notebook:

```bash
python generate_report.py \
    --json "C:\datos\_annotations.coco.json" \
    --images_dir "C:\datos\images" \
    --output "mi_reporte.html" \
    --n_steps 4 \
    --model biomedclip \
    --preprocessing clahe
```